In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
df = pd.read_csv('/Users/giahuy/Documents/GitHub/topicmodeling/tm_research/data/processed/train_preference_curated.csv')

# Basic info about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst few rows:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Basic statistics
print("\nBasic statistics:")
print(df.describe())

# Check distributions of categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print("\nCategorical column distributions:")
    for col in categorical_cols:
        print(f"\n{col}:")
        print(df[col].value_counts())

# Check distributions of numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 0:
    print("\nNumerical column distributions:")
    
    # Create subplots for histograms
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    if len(numerical_cols) > 0:
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, col in enumerate(numerical_cols):
            if i < len(axes):
                axes[i].hist(df[col].dropna(), bins=30, alpha=0.7)
                axes[i].set_title(f'Distribution of {col}')
                axes[i].set_xlabel(col)
                axes[i].set_ylabel('Frequency')
        
        # Hide empty subplots
        for i in range(len(numerical_cols), len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()

# Create correlation matrix if there are numerical columns
if len(numerical_cols) > 1:
    plt.figure(figsize=(10, 8))
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Correlation Matrix')
    plt.show()


Dataset shape: (6997, 3)

Column names:
['Emotion', 'Sentence', 'emotion_vn']

Data types:
Emotion       object
Sentence      object
emotion_vn    object
dtype: object

First few rows:
  Emotion                                           Sentence emotion_vn
0   Other              cho mình xin bài nhạc tên là gì với ạ       khác
1   Other                      một lí do trog muôn vàn lí do       khác
2   Other  trời nắng nóng thế này mình muốn bán nước khôn...       khác
3   Other                       bếp dầu , nhiều nhà vẫn dùng       khác
4   Other  nếu thấy phụ nữ quá phức tạp để hiểu và chinh ...       khác

Missing values:
Emotion          0
Sentence         0
emotion_vn    1449
dtype: int64

Basic statistics:
          Emotion              Sentence emotion_vn
count        6997                  6997       5548
unique          7                  6977          7
top     Enjoyment  Mai thi rồi, sợ qué.     vui vẻ
freq         1558                     4       1558

Categorical column di

In [4]:
# Fix missing Vietnamese emotion labels
print("Fixing missing Vietnamese emotion labels...")

# Create a mapping dictionary for English to Vietnamese emotions
emotion_mapping = {
    'Enjoyment': 'vui vẻ',
    'Disgust': 'khó chịu', 
    'Other': 'khác',
    'Sadness': 'buồn',
    'Anger': 'giận dữ',
    'Surprise': 'ngạc nhiên',
    'Fear': 'sợ hãi'
}

# Check missing values before fixing
print(f"Missing values in emotion_vn before fixing: {df['emotion_vn'].isna().sum()}")

# Fill missing Vietnamese emotion labels using the English emotion column
df['emotion_vn'] = df['emotion_vn'].fillna(df['Emotion'].map(emotion_mapping))

# Verify the fix
print(f"Missing values in emotion_vn after fixing: {df['emotion_vn'].isna().sum()}")

# Check the updated distribution
print("\nUpdated emotion_vn distribution:")
print(df['emotion_vn'].value_counts().sort_index())

# Verify that English and Vietnamese emotions now match
print("\nVerification - English vs Vietnamese emotion counts:")
english_counts = df['Emotion'].value_counts().sort_index()
vietnamese_counts = df['emotion_vn'].value_counts().sort_index()

comparison_df = pd.DataFrame({
    'English_Count': english_counts,
    'Vietnamese_Count': vietnamese_counts
})
print(comparison_df)

# Check if they match perfectly
matches = (comparison_df['English_Count'] == comparison_df['Vietnamese_Count']).all()
print(f"\nDo all emotion counts match? {matches}")


Fixing missing Vietnamese emotion labels...
Missing values in emotion_vn before fixing: 1449
Missing values in emotion_vn after fixing: 0

Updated emotion_vn distribution:
emotion_vn
buồn           947
giận dữ        800
khác          1021
khó chịu      1071
ngạc nhiên     800
sợ hãi         800
vui vẻ        1558
Name: count, dtype: int64

Verification - English vs Vietnamese emotion counts:
            English_Count  Vietnamese_Count
Anger               800.0               NaN
Disgust            1071.0               NaN
Enjoyment          1558.0               NaN
Fear                800.0               NaN
Other              1021.0               NaN
Sadness             947.0               NaN
Surprise            800.0               NaN
buồn                  NaN             947.0
giận dữ               NaN             800.0
khác                  NaN            1021.0
khó chịu              NaN            1071.0
ngạc nhiên            NaN             800.0
sợ hãi                NaN       

In [7]:
import os
# Export the cleaned dataset to CSV
print("Exporting cleaned dataset to CSV...")

# Define the output directory and filename
output_dir = '/Users/giahuy/Documents/GitHub/topicmodeling/tm_research/data/processed'
output_filename = 'cleaned_emotion_dataset.csv'
output_path = os.path.join(output_dir, output_filename)

# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Export to CSV
df.to_csv(output_path, index=False, encoding='utf-8')

print(f"Dataset successfully exported to: {output_path}")
print(f"Total rows exported: {len(df)}")
print(f"Total columns exported: {len(df.columns)}")


Exporting cleaned dataset to CSV...
Dataset successfully exported to: /Users/giahuy/Documents/GitHub/topicmodeling/tm_research/data/processed/cleaned_emotion_dataset.csv
Total rows exported: 6997
Total columns exported: 3
